In [ ]:
"""
Snowflake UDTF that runs ``run_pf_batch_fast`` on a pickled LFE Network
with time-series load data, distributing work across partitions via a
Snowflake compute pool.

Usage
-----
Call ``LFEBatchFastPowerflow.register_udtf(session, …)`` from a stored
procedure to register the UDTF, then invoke it from SQL (see the
companion ``lfe_batch_fast_udtf.sql``).
"""

import logging
import pickle
import time

import numpy as np
import pandas as pd
import snowflake.snowpark

from load_flow_engine.time_series import run_pf_batch_fast
from snowflake.snowpark.functions import PandasDataFrameType, pandas_udtf
from snowflake.snowpark.types import (
    BinaryType,
    BooleanType,
    DoubleType,
    IntegerType,
    PandasDataFrame,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --------------------------------------------------------------------- #
# Schema constants                                                       #
# --------------------------------------------------------------------- #

INPUT_COLS = [
    "CIRCUIT_KEY",          # VARCHAR  – circuit identifier
    "NETWORK_BLOB",         # BINARY   – pickled load_flow_engine.Network
    "REPORTED_DTTM",        # TIMESTAMP – hour timestamp
    "MEASURE_VALUE",        # DOUBLE   – total active power (MW) for this hour
    "PARTITION_ID",         # INTEGER  – which partition this row belongs to
]

OUTPUT_FIELDS = [
    StructField("CIRCUIT_KEY",   StringType(),    nullable=False),
    StructField("PARTITION_ID",  IntegerType(),   nullable=False),
    StructField("REPORTED_DTTM", StringType(),    nullable=True),
    StructField("BUS_ID",        StringType(),    nullable=True),
    StructField("BUS_NAME",      StringType(),    nullable=True),
    StructField("VM_PU",         DoubleType(),    nullable=True),
    StructField("VA_DEGREE",     DoubleType(),    nullable=True),
    StructField("V_A_PU",        DoubleType(),    nullable=True),
    StructField("V_B_PU",        DoubleType(),    nullable=True),
    StructField("V_C_PU",        DoubleType(),    nullable=True),
    StructField("VA_A_DEGREE",   DoubleType(),    nullable=True),
    StructField("VA_B_DEGREE",   DoubleType(),    nullable=True),
    StructField("VA_C_DEGREE",   DoubleType(),    nullable=True),
    StructField("CONVERGED",     BooleanType(),   nullable=False),
    StructField("HR",            IntegerType(),   nullable=True),
    StructField("SOLVE_TIME_SEC", DoubleType(),   nullable=True),
    StructField("ERROR_MESSAGE", StringType(),    nullable=True),
]


# --------------------------------------------------------------------- #
# Registration wrapper                                                   #
# --------------------------------------------------------------------- #

class LFEBatchFastPowerflow:
    """Registers and manages the ``LFE_BATCH_FAST_PF`` Snowflake UDTF."""

    def __init__(
        self,
        session: snowflake.snowpark.Session,
        database: str,
        schema: str,
        stage_name: str,
    ):
        self.session = session
        self.database = database
        self.schema = schema
        self.stage_name = stage_name

    def register_udtf(
        self,
        function_name: str = "LFE_BATCH_FAST_PF",
    ) -> str:
        stage_location = f"{self.database}.{self.schema}.{self.stage_name}"
        qualified_name = (
            function_name
            if "." in function_name
            else f"{self.database}.{self.schema}.{function_name}"
        )

        @pandas_udtf(
            packages=(
                "snowflake-snowpark-python",
                "numpy==1.26.4",
                "pandas==2.3.2",
                "scipy==1.15.3",
                "typing_extensions==4.15.0",
            ),
            name=qualified_name,
            replace=True,
            stage_location=stage_location,
            output_schema=StructType(OUTPUT_FIELDS),
            input_types=[
                PandasDataFrameType([
                    StringType(),       # CIRCUIT_KEY
                    BinaryType(),       # NETWORK_BLOB
                    TimestampType(),    # REPORTED_DTTM
                    DoubleType(),       # MEASURE_VALUE
                    IntegerType(),      # PARTITION_ID
                ])
            ],
            input_names=INPUT_COLS,
            statement_params={
                "STATEMENT_TIMEOUT_IN_SECONDS": 7200,
                "PYTHON_UDTF_END_PARTITION_TIMEOUT_SECONDS": 3600,
            },
            is_permanent=True,
        )
        class LFEBatchFastPFUDTF:
            """
            Each partition receives all time-series rows for ONE
            (CIRCUIT_KEY, PARTITION_ID) combination.  The UDTF:

            1. Deserialises the Network blob (same blob repeated per row).
            2. Sorts the load values by timestamp.
            3. Calls ``run_pf_batch_fast`` on the sorted MW series.
            4. Maps the ``hr`` index back to the original timestamps.
            5. Returns per-bus, per-hour voltage results.
            """

            def end_partition(self, df: PandasDataFrame) -> pd.DataFrame:
                circuit_key = ""
                partition_id = 0
                try:
                    if df.empty:
                        return pd.DataFrame(
                            columns=[f.name for f in OUTPUT_FIELDS]
                        )

                    circuit_key = str(
                        df["CIRCUIT_KEY"].dropna().iloc[0]
                    )
                    partition_id = int(
                        df["PARTITION_ID"].dropna().iloc[0]
                    )

                    # --- Deserialise network ---
                    blob = df["NETWORK_BLOB"].dropna().iloc[0]
                    net = pickle.loads(blob)

                    # --- Build ordered MW series ---
                    ts = (
                        df[["REPORTED_DTTM", "MEASURE_VALUE"]]
                        .dropna(subset=["MEASURE_VALUE"])
                        .sort_values("REPORTED_DTTM")
                        .reset_index(drop=True)
                    )

                    if ts.empty:
                        return pd.DataFrame([{
                            "CIRCUIT_KEY": circuit_key,
                            "PARTITION_ID": partition_id,
                            "REPORTED_DTTM": None,
                            "BUS_ID": None,
                            "BUS_NAME": None,
                            "VM_PU": None,
                            "VA_DEGREE": None,
                            "V_A_PU": None,
                            "V_B_PU": None,
                            "V_C_PU": None,
                            "VA_A_DEGREE": None,
                            "VA_B_DEGREE": None,
                            "VA_C_DEGREE": None,
                            "CONVERGED": False,
                            "HR": None,
                            "SOLVE_TIME_SEC": None,
                            "ERROR_MESSAGE": "No valid MEASURE_VALUE rows",
                        }])

                    values = ts["MEASURE_VALUE"].values.astype(float)
                    timestamps = ts["REPORTED_DTTM"].values

                    # --- Run fast batch power flow ---
                    t0 = time.perf_counter()
                    result_df = run_pf_batch_fast(
                        net,
                        values,
                        warm_start=True,
                        power_factor=0.95,
                    )
                    solve_time = time.perf_counter() - t0

                    # --- Map hr back to timestamp ---
                    hr_to_ts = {
                        hr + 1: timestamps[hr]
                        for hr in range(len(timestamps))
                    }
                    result_df = result_df.reset_index()
                    result_df["REPORTED_DTTM"] = (
                        result_df["hr"].map(hr_to_ts).astype(str)
                    )

                    # --- Build output ---
                    result_df["CIRCUIT_KEY"] = circuit_key
                    result_df["PARTITION_ID"] = partition_id
                    result_df["BUS_ID"] = result_df["bus"].astype(str)
                    result_df["BUS_NAME"] = result_df["name"].fillna("")
                    result_df["CONVERGED"] = result_df["converged"]
                    result_df["HR"] = result_df["hr"]
                    result_df["SOLVE_TIME_SEC"] = solve_time
                    result_df["ERROR_MESSAGE"] = None

                    return result_df.rename(columns={
                        "vm_pu":        "VM_PU",
                        "va_degree":    "VA_DEGREE",
                        "v_a_pu":       "V_A_PU",
                        "v_b_pu":       "V_B_PU",
                        "v_c_pu":       "V_C_PU",
                        "va_a_degree":  "VA_A_DEGREE",
                        "va_b_degree":  "VA_B_DEGREE",
                        "va_c_degree":  "VA_C_DEGREE",
                    })[[f.name for f in OUTPUT_FIELDS]]

                except Exception as exc:
                    logger.exception(
                        "LFEBatchFastPFUDTF failed for circuit=%s partition=%s",
                        circuit_key, partition_id,
                    )
                    return pd.DataFrame([{
                        "CIRCUIT_KEY": circuit_key or "",
                        "PARTITION_ID": partition_id,
                        "REPORTED_DTTM": None,
                        "BUS_ID": None,
                        "BUS_NAME": None,
                        "VM_PU": None,
                        "VA_DEGREE": None,
                        "V_A_PU": None,
                        "V_B_PU": None,
                        "V_C_PU": None,
                        "VA_A_DEGREE": None,
                        "VA_B_DEGREE": None,
                        "VA_C_DEGREE": None,
                        "CONVERGED": False,
                        "HR": None,
                        "SOLVE_TIME_SEC": None,
                        "ERROR_MESSAGE": str(exc),
                    }])

        logger.info("Registered LFE batch fast PF UDTF: %s", qualified_name)
        return qualified_name


In [ ]:
-- ============================================================
-- LFE_BATCH_FAST_PF  –  distributed time-series power flow
-- ============================================================
--
-- This UDTF runs ``run_pf_batch_fast`` inside Snowflake on a
-- compute pool.  The caller chooses how many time-series
-- partitions to split the workload into (N_PARTITIONS), and
-- Snowflake distributes each (CIRCUIT_KEY, PARTITION_ID) pair
-- to a separate worker node in the pool.
--
-- Inputs
-- ------
--   ENCODED_TABLE  – table with pickled Network blobs
--                    (e.g. NMM_F_TOPOLOGICALNODE_ENCODED_CIRCUITS_C)
--   TS_TABLE       – table with hourly MW readings
--                    (e.g. NMM_F_HIST_HR_GROSS_DERGEN_DLY_C_PP_VW)
--   N_PARTITIONS   – number of time-series chunks per circuit
--
-- ============================================================


-- 1. Output table ---------------------------------------------------
CREATE TABLE IF NOT EXISTS TESTDB.PUBLIC.LFE_BATCH_FAST_PF_RESULTS (
    CIRCUIT_KEY     VARCHAR  NOT NULL,
    PARTITION_ID    INTEGER  NOT NULL,
    REPORTED_DTTM   VARCHAR,
    BUS_ID          VARCHAR,
    BUS_NAME        VARCHAR,
    VM_PU           DOUBLE,
    VA_DEGREE       DOUBLE,
    V_A_PU          DOUBLE,
    V_B_PU          DOUBLE,
    V_C_PU          DOUBLE,
    VA_A_DEGREE     DOUBLE,
    VA_B_DEGREE     DOUBLE,
    VA_C_DEGREE     DOUBLE,
    CONVERGED       BOOLEAN  NOT NULL,
    HR              INTEGER,
    SOLVE_TIME_SEC  DOUBLE,
    ERROR_MESSAGE   VARCHAR
);


-- 2. Stored procedure: register the Python UDTF -----------------------
--    Run this once (or after code changes) to push the UDTF into
--    the stage and make it callable.

CREATE OR REPLACE PROCEDURE TESTDB.PUBLIC.SP_REGISTER_LFE_BATCH_FAST_PF()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = (
    'snowflake-snowpark-python',
    'numpy==1.26.4',
    'pandas==2.3.2',
    'scipy==1.15.3',
    'typing_extensions==4.15.0'
)
IMPORTS = ('@TESTDB.PUBLIC.SERVICE_STAGE/lfe_batch_fast_udtf.py',
           '@TESTDB.PUBLIC.SERVICE_STAGE/load_flow_engine.zip')
HANDLER = 'register_handler'
EXECUTE AS CALLER
AS
$$
def register_handler(session):
    from lfe_batch_fast_udtf import LFEBatchFastPowerflow
    wrapper = LFEBatchFastPowerflow(
        session,
        database="TESTDB",
        schema="PUBLIC",
        stage_name="SERVICE_STAGE",
    )
    return wrapper.register_udtf("TESTDB.PUBLIC.LFE_BATCH_FAST_PF")
$$;

-- Register the UDTF
CALL TESTDB.PUBLIC.SP_REGISTER_LFE_BATCH_FAST_PF();


-- 3. Example: run on a compute pool with N partitions -----------------
--
--    The CTE assigns each hourly row to one of N_PARTITIONS buckets
--    using NTILE().  Each (CIRCUIT_KEY, PARTITION_ID) combination
--    becomes a separate UDTF partition → separate compute node.
--
--    Adjust :n_partitions (e.g. 4, 8, 16) based on cluster size.

SET n_partitions = 4;

INSERT INTO TESTDB.PUBLIC.LFE_BATCH_FAST_PF_RESULTS
WITH partitioned_ts AS (
    SELECT
        ts.STRUCT_NUM                           AS CIRCUIT_KEY,
        ts.REPORTED_DTTM,
        ts.MEASURE_VALUE,
        NTILE($n_partitions) OVER (
            PARTITION BY ts.STRUCT_NUM
            ORDER BY ts.REPORTED_DTTM
        )                                       AS PARTITION_ID
    FROM TESTDB.PUBLIC.NMM_F_HIST_HR_GROSS_DERGEN_DLY_C_PP_VW ts
    WHERE ts.STRUCT_NUM = 'CKT_4799_01085'         -- filter to target circuit(s)
),
source AS (
    SELECT
        pts.CIRCUIT_KEY,
        enc.PRELOAD_ENCODED_CA                  AS NETWORK_BLOB,
        pts.REPORTED_DTTM,
        pts.MEASURE_VALUE,
        pts.PARTITION_ID
    FROM partitioned_ts pts
    JOIN TESTDB.PUBLIC.NMM_F_TOPOLOGICALNODE_ENCODED_CIRCUITS_C enc
        ON enc.CIRCUIT_KEY = pts.CIRCUIT_KEY
)
SELECT
    pf.CIRCUIT_KEY,
    pf.PARTITION_ID,
    pf.REPORTED_DTTM,
    pf.BUS_ID,
    pf.BUS_NAME,
    pf.VM_PU,
    pf.VA_DEGREE,
    pf.V_A_PU,
    pf.V_B_PU,
    pf.V_C_PU,
    pf.VA_A_DEGREE,
    pf.VA_B_DEGREE,
    pf.VA_C_DEGREE,
    pf.CONVERGED,
    pf.HR,
    pf.SOLVE_TIME_SEC,
    pf.ERROR_MESSAGE
FROM source,
     TABLE(
         TESTDB.PUBLIC.LFE_BATCH_FAST_PF(
             source.CIRCUIT_KEY,
             source.NETWORK_BLOB,
             source.REPORTED_DTTM,
             source.MEASURE_VALUE,
             source.PARTITION_ID
         ) OVER (PARTITION BY source.CIRCUIT_KEY, source.PARTITION_ID)
     ) pf;


-- 4. Run on a compute pool -------------------------------------------
--    Execute the above query using a Snowpark Container Services
--    compute pool for true parallel compute-node distribution.
--
--    Option A: EXECUTE SERVICE on a compute pool
--    (wrap the INSERT in a stored procedure and execute on pool)

CREATE OR REPLACE PROCEDURE TESTDB.PUBLIC.SP_RUN_BATCH_FAST_PF(
    encoded_table VARCHAR,
    ts_table VARCHAR,
    n_partitions INTEGER
)
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run_handler'
EXECUTE AS CALLER
AS
$$
def run_handler(session, encoded_table: str, ts_table: str, n_partitions: int):
    query = f"""
    INSERT INTO TESTDB.PUBLIC.LFE_BATCH_FAST_PF_RESULTS
    WITH partitioned_ts AS (
        SELECT
            ts.STRUCT_NUM                           AS CIRCUIT_KEY,
            ts.REPORTED_DTTM,
            ts.MEASURE_VALUE,
            NTILE({int(n_partitions)}) OVER (
                PARTITION BY ts.STRUCT_NUM
                ORDER BY ts.REPORTED_DTTM
            )                                       AS PARTITION_ID
        FROM {ts_table} ts
    ),
    source AS (
        SELECT
            pts.CIRCUIT_KEY,
            enc.PRELOAD_ENCODED_CA                  AS NETWORK_BLOB,
            pts.REPORTED_DTTM,
            pts.MEASURE_VALUE,
            pts.PARTITION_ID
        FROM partitioned_ts pts
        JOIN {encoded_table} enc
            ON enc.CIRCUIT_KEY = pts.CIRCUIT_KEY
    )
    SELECT
        pf.CIRCUIT_KEY,
        pf.PARTITION_ID,
        pf.REPORTED_DTTM,
        pf.BUS_ID,
        pf.BUS_NAME,
        pf.VM_PU,
        pf.VA_DEGREE,
        pf.V_A_PU,
        pf.V_B_PU,
        pf.V_C_PU,
        pf.VA_A_DEGREE,
        pf.VA_B_DEGREE,
        pf.VA_C_DEGREE,
        pf.CONVERGED,
        pf.HR,
        pf.SOLVE_TIME_SEC,
        pf.ERROR_MESSAGE
    FROM source,
         TABLE(
             TESTDB.PUBLIC.LFE_BATCH_FAST_PF(
                 source.CIRCUIT_KEY,
                 source.NETWORK_BLOB,
                 source.REPORTED_DTTM,
                 source.MEASURE_VALUE,
                 source.PARTITION_ID
             ) OVER (PARTITION BY source.CIRCUIT_KEY, source.PARTITION_ID)
         ) pf
    """
    session.sql(query).collect()
    return f"SUCCESS: {n_partitions} partitions dispatched"
$$;


-- 5. Execute on a compute pool ----------------------------------------
--    This runs the stored procedure on the RAY_POC compute pool,
--    which distributes the UDTF partitions across pool nodes.

EXECUTE SERVICE
    IN COMPUTE POOL RAY_POC
    FROM @TESTDB.PUBLIC.SERVICE_STAGE
    SPEC = 'lfe_batch_fast_spec.yaml'
    QUERY_WAREHOUSE = TEST_SMALL;

-- Or invoke the stored procedure directly (warehouse-based):
CALL TESTDB.PUBLIC.SP_RUN_BATCH_FAST_PF(
    'TESTDB.PUBLIC.NMM_F_TOPOLOGICALNODE_ENCODED_CIRCUITS_C',
    'TESTDB.PUBLIC.NMM_F_HIST_HR_GROSS_DERGEN_DLY_C_PP_VW',
    4   -- number of partitions
);


In [ ]:
import pickle
import snowflake.snowpark

import types
import multiconductor as mc

from powerflow_pipeline.powerflow import ExcludedDevices, Pipeline, ResultHandler
from powerflow_pipeline.util import PowerflowConfig
from powerflow_snowflake.ami_load_allocation_wrapper import AMILoadAllocation
from powerflow_snowflake.batch_runner import SnowflakePowerflowRunner
from powerflow_snowflake.circuit_cache_builder import SnowflakeCircuitCacheManager
from powerflow_lfe.lfe_load_flow_wrapper import LFELoadFlow
from powerflow_lfe.lfe_batch_fast_udtf import LFEBatchFastPowerflow
from powerflow_snowflake.load_allocation_wrapper import LoadAllocation
from sce.load_allocation import SCELoadAllocationMeasurement
from sce.sce_mc_mapping import SCECircuitDataTransformer, SCELoadProfileController, SCEResultsTransformer, SCEPowerflowPipelineTemp, SnowflakeDataSource
import logging

from .circuit_encoder import CircuitEncoder

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 
def run_load_allocation(
    session: snowflake.snowpark.Session,
    pf_config_json: str,
    start_time: str,
    end_time: str,
    circuit_keys: str,
    procedure: str,
):
    if procedure == 'CC_ALLOCATION':
        load_allocation = LoadAllocation(
            session,
            pf_config_json,
            start_time,
            end_time,
            circuit_keys
        )
        run_tag = load_allocation.run_load_allocation()
    elif procedure == 'AMI_ALLOCATION':
        load_allocation = AMILoadAllocation(
            session,
            pf_config_json,
            start_time,
            end_time,
            circuit_keys
        )
        run_tag = load_allocation.run_load_allocation()
    return (run_tag)

def run_powerflow(
    session: snowflake.snowpark.Session,
    pf_config_json: str,
    start_time: str,
    end_time: str,
    circuit_keys: str,
    circuit_size: str,
):
    runner = SnowflakePowerflowRunner(
        session,
        pf_config_json,
        start_time,
        end_time,
        circuit_keys,
        circuit_size,
    )

    run_tag = runner.run_pf_batches()
    return (run_tag)


def register_lfe_load_flow_udtf(
    session: snowflake.snowpark.Session,
    pf_config_json: str,
    function_name: str,
):
    wrapper = LFELoadFlow(session, pf_config_json)
    return wrapper.register_udtf(function_name or "LFE_LOAD_FLOW")


def register_lfe_batch_fast_udtf(
    session: snowflake.snowpark.Session,
    database: str,
    schema: str,
    stage_name: str,
    function_name: str = "LFE_BATCH_FAST_PF",
):
    wrapper = LFEBatchFastPowerflow(session, database, schema, stage_name)
    return wrapper.register_udtf(function_name)


def define_circuit_parallel(
    session: snowflake.snowpark.Session,
    circuit_key: str,
    pf_config_json: str,
    capacitor_enabled: bool,
    generator_enabled: bool,
    transformer_enabled: bool,
    regulator_enabled: bool,
) -> str:
    circuit_keys = [c.strip() for c in circuit_key.split(',')]
    pf_config = PowerflowConfig()
    pf_config.load_config_from_json(pf_config_json)
    logger.info(f"define_circuit::multiconductor version: {mc.__version__}")
    circuit_encoder = CircuitEncoder(session=session, pf_config=pf_config, circuit_keys=circuit_key, capacitor_enabled=capacitor_enabled,
                                     generator_enabled=generator_enabled, transformer_enabled=transformer_enabled, regulator_enabled=regulator_enabled)
    circuit_encoder.encode_circuits()


def define_circuit(
    session: snowflake.snowpark.Session,
    circuit_key: str,
    pf_config_json: str,
    capacitor_enabled: bool,
    generator_enabled: bool,
    transformer_enabled: bool,
    regulator_enabled: bool,
) -> str:
    circuit_keys = [c.strip() for c in circuit_key.split(',')]
    pf_config = PowerflowConfig()
    pf_config.load_config_from_json(pf_config_json)
    logger.info(f"define_circuit::multiconductor version: {mc.__version__}")

    result_handler = ResultHandler(result_transformer=SCEResultsTransformer())
    for ck in circuit_keys:
        data_source = SnowflakeDataSource(
            session,
            pf_config.get_database(),
            pf_config.get_database_schema(),
            pf_config.get_bus_table_database(),
            pf_config.get_bus_table_schema(),
            pf_config.get_connectivity_table_database(),
            pf_config.get_connectivity_table_schema(),
            pf_config.get_bus_table(),
            pf_config.get_connectivity_table(),
            ck,
        )

        network_transformer = SCECircuitDataTransformer()
        load_profile_controller = SCELoadProfileController()
        load_allocation_meas = SCELoadAllocationMeasurement()

        pipeline = SCEPowerflowPipelineTemp(
            data_source,
            transformer=network_transformer,
            load_allocation_measurement=load_allocation_meas,
            result_handler=result_handler,
            load_profile_controller=load_profile_controller,
        )
        excluded_devices = ExcludedDevices()

        if not capacitor_enabled:
            excluded_devices.disable_shunts()
        if not generator_enabled:
            excluded_devices.disable_asymmetric_sgens()
        if not transformer_enabled:
            excluded_devices.disable_transformers()
        if not regulator_enabled:
            excluded_devices.disable_regulators()

        pipeline.set_excluded_devices(excluded_devices)

        sf_cache_manager = SnowflakeCircuitCacheManager(session, pf_config)
        logger.info(f"Generating net for circuit {ck}")
        sf_cache_manager.put_circuit_object(
            ck,
            pipeline
        )

    return f"SUCCESS ({len(circuit_keys)})"
